# Notebook 08 — GATK HaplotypeCaller: Concepts and Architecture

**Module:** 09 — Genomics and NGS  
**Notebook:** 08 of 16  
**Time estimate:** 75 minutes

> HaplotypeCaller is the GATK standard for germline variant calling.
> Know the three-phase architecture (active regions → assembly → PairHMM)
> and the GVCF workflow for joint genotyping.

---
## Step 1 — Motivation

Simple pileup callers (NB07) fail at indels: an insertion or deletion causes reads
to misalign, creating apparent mismatches at flanking positions (false SNPs) while
masking the true indel. HaplotypeCaller solves this by locally reassembling the
reads at active regions, discovering all variants simultaneously.

---
## Step 2 — Intuition

**Three-phase HaplotypeCaller architecture:**

**Phase 1 — Active region detection:**  
Slide a window across the BAM. If the local read activity (mismatches, indels)
exceeds a threshold, mark that region as "active." Active regions are assembled
de novo; the rest are handled by simple reference model calls.

**Phase 2 — Local de novo assembly:**  
Within each active region, build a De Bruijn graph from the reads (just like NB14
in Module 08). Traverse the graph to find all plausible haplotypes — alternative
sequences containing combinations of variants. This handles MNPs, indels, and
linked variants simultaneously.

**Phase 3 — PairHMM genotyping:**  
For each read and each candidate haplotype, compute the probability that the read
came from that haplotype using a Pair Hidden Markov Model. These per-haplotype
likelihoods feed into the genotype likelihood calculation.

---
## Step 3 — Biological Background

**Why local reassembly is necessary for indels:**

A 3 bp deletion in a sample causes reads that span the deletion to be
soft-clipped or misaligned by BWA. The resulting pileup shows:
- Reads ending prematurely (soft clips)
- Downstream bases appearing shifted → false SNPs at every position
- The deletion itself may not appear in the pileup at all

By reassembling the reads de novo, HaplotypeCaller reconstructs:
- Haplotype 1: reference sequence (no deletion)
- Haplotype 2: sequence with the 3 bp deletion
Then it aligns reads to both haplotypes — reads that span the deletion align
correctly to haplotype 2.

**GVCF mode and joint genotyping:**
- **GVCF (genomic VCF):** Contains variant calls AND reference confidence scores
  at every position. Critical for multi-sample studies — allows adding samples
  later without re-running all samples.
- **GenomicsDBImport:** Combines per-sample GVCFs into a database
- **GenotypeGVCFs:** Joint genotyping from the combined database, leveraging
  information across samples to improve genotype calls

---
## Step 4 — Mathematical Explanation

**Active region scoring:**
At each position, compute the activity score:
$$A_i = \sum_{r \in \text{reads covering } i} \mathbb{1}[b_r(i) \neq \text{ref}_i \text{ and } Q_{r,i} \geq 20]$$

If $A_i > \text{threshold}$, include position $i$ in the active region.

**PairHMM:** A pair HMM aligns a read $R$ to a haplotype $H$ probabilistically,
allowing sequencing errors, insertions, and deletions. States:
- **M:** match/mismatch between $R$ and $H$
- **I:** insertion in $R$ relative to $H$
- **D:** deletion in $R$ relative to $H$ (base in $H$ not in $R$)

The HMM computes:
$$P(R \mid H) = \sum_{\text{all alignments}} P(R, \text{alignment} \mid H)$$

This marginal probability over all alignments (via forward algorithm) accounts for
alignment uncertainty in a principled way.

**Genotype likelihood with haplotypes:**
$$P(\text{reads} \mid \text{genotype } H_1/H_2) = \prod_{r} \frac{P(r \mid H_1) + P(r \mid H_2)}{2}$$

In [ ]:
# Step 6 — Python: active region detection and De Bruijn graph assembly
import numpy as np
from collections import defaultdict


def detect_active_regions(
    pileup: dict,  # {pos: {'bases': list, 'quals': list, 'ref': str}}
    window: int = 50,
    min_activity: float = 0.2,
    min_qual: int = 20
) -> list[tuple[int, int]]:
    """
    Slide a window; mark regions where per-base mismatch rate > min_activity.
    Returns list of (start, end) active region tuples.
    """
    positions = sorted(pileup.keys())
    activity = {}
    for pos in positions:
        p = pileup[pos]
        ref = p['ref']
        n_mismatch = sum(
            1 for b, q in zip(p['bases'], p['quals'])
            if b != ref and q >= min_qual
        )
        depth = len(p['bases'])
        activity[pos] = n_mismatch / depth if depth > 0 else 0

    # Smooth with sliding window
    active_regions = []
    in_region = False
    region_start = None

    for pos in positions:
        window_positions = [p for p in positions if pos <= p < pos + window]
        mean_activity = np.mean([activity.get(p, 0) for p in window_positions])

        if mean_activity >= min_activity and not in_region:
            in_region = True
            region_start = pos
        elif mean_activity < min_activity and in_region:
            in_region = False
            active_regions.append((region_start, pos + window))

    if in_region:
        active_regions.append((region_start, positions[-1] + window))

    return active_regions


def build_debruijn(reads: list[str], k: int = 11) -> dict[str, list[str]]:
    """Build De Bruijn graph from reads. Same as Module 08 NB14."""
    graph = defaultdict(list)
    for read in reads:
        for i in range(len(read) - k + 1):
            kmer = read[i:i+k]
            prefix = kmer[:-1]
            suffix = kmer[1:]
            if suffix not in graph[prefix]:
                graph[prefix].append(suffix)
    return dict(graph)


def find_haplotypes_greedy(graph: dict, source: str, max_paths: int = 4) -> list[str]:
    """Find up to max_paths paths through De Bruijn graph from source."""
    paths = []
    stack = [(source, source)]  # (current_node, path_so_far)
    while stack and len(paths) < max_paths:
        node, path = stack.pop()
        neighbors = graph.get(node, [])
        if not neighbors:
            paths.append(path)
        else:
            for neighbor in neighbors[:2]:  # limit branching
                stack.append((neighbor, path + neighbor[-1]))
    return paths


# Simulate an active region with a 2 bp deletion
ref_segment = 'ATCGATCGATCGATCGATCG'   # 20 bp reference
alt_segment = 'ATCGATCG' + 'ATCGATCG'  # same length but different repeat
# Simulate reads: 10 from ref, 10 from alt (heterozygous deletion)
rng = np.random.default_rng(42)

def make_reads(template: str, n: int = 10, read_len: int = 15, error_rate: float = 0.01) -> list[str]:
    reads = []
    for _ in range(n):
        start = rng.integers(0, max(1, len(template) - read_len + 1))
        read = list(template[start:start + read_len])
        for j in range(len(read)):
            if rng.random() < error_rate:
                read[j] = rng.choice([b for b in 'ACGT' if b != read[j]])
        reads.append(''.join(read))
    return reads


ref_reads = make_reads(ref_segment, n=10, read_len=15)
alt_reads = make_reads(alt_segment, n=10, read_len=15)
all_reads = ref_reads + alt_reads

print("Reads in active region:")
for r in all_reads[:5]: print(f"  {r}")

# Build De Bruijn graph
graph = build_debruijn(all_reads, k=7)
print(f"\nDe Bruijn graph nodes: {len(graph)}")
print(f"Sample edges:")
for k, vs in list(graph.items())[:3]:
    print(f"  {k} → {vs}")

In [ ]:
# Simplified PairHMM: compute P(read | haplotype)
def pair_hmm_log_prob(
    read: str,
    haplotype: str,
    match_prob: float = 0.98,
    gap_open: float = 0.002,
    gap_extend: float = 0.3
) -> float:
    """
    Simplified PairHMM: compute log10 P(read | haplotype).
    Uses DP over match/insert/delete states.
    This is a teaching implementation; real GATK PairHMM is heavily optimized.
    """
    n, m = len(read), len(haplotype)
    NEG_INF = -1e9

    # Three DP tables: M (match), I (insert in read), D (delete from read)
    M = np.full((n+1, m+1), NEG_INF)
    I = np.full((n+1, m+1), NEG_INF)
    D = np.full((n+1, m+1), NEG_INF)

    # Initialization: allow free start at any haplotype position
    for j in range(m+1):
        M[0][j] = np.log10(1.0 / (m + 1))  # uniform prior over starting position

    mismatch_prob = (1 - match_prob) / 3

    for i in range(1, n+1):
        for j in range(1, m+1):
            emit = np.log10(match_prob if read[i-1] == haplotype[j-1] else mismatch_prob)
            # Match state
            from_m = M[i-1][j-1] if M[i-1][j-1] > NEG_INF else NEG_INF
            from_i = I[i-1][j-1] if I[i-1][j-1] > NEG_INF else NEG_INF
            from_d = D[i-1][j-1] if D[i-1][j-1] > NEG_INF else NEG_INF
            best_prev = max(from_m, from_i, from_d)
            M[i][j] = emit + best_prev if best_prev > NEG_INF else NEG_INF
            # Insert state (base in read, not in haplotype)
            I[i][j] = max(
                M[i-1][j] + np.log10(gap_open) if M[i-1][j] > NEG_INF else NEG_INF,
                I[i-1][j] + np.log10(gap_extend) if I[i-1][j] > NEG_INF else NEG_INF
            )
            # Delete state (base in haplotype, not in read)
            D[i][j] = max(
                M[i][j-1] + np.log10(gap_open) if M[i][j-1] > NEG_INF else NEG_INF,
                D[i][j-1] + np.log10(gap_extend) if D[i][j-1] > NEG_INF else NEG_INF
            )

    # Final: sum over all end states in last row
    last_row = np.concatenate([M[n], I[n], D[n]])
    valid = last_row[last_row > NEG_INF]
    if len(valid) == 0:
        return NEG_INF
    # Log-sum-exp
    max_val = valid.max()
    return max_val + np.log10(np.sum(10 ** (valid - max_val)))


# Test: align ref and alt reads to both haplotypes
print("PairHMM: P(read | haplotype)")
print(f"  Reference haplotype: {ref_segment}")
print(f"  Alternate haplotype: {alt_segment}")
print()
print(f"  {'Read':20s}  {'log P(read|REF)':>16}  {'log P(read|ALT)':>16}  {'Prefers':>8}")
print('-' * 70)
for r in all_reads[:6]:
    p_ref = pair_hmm_log_prob(r, ref_segment)
    p_alt = pair_hmm_log_prob(r, alt_segment)
    pref = 'REF' if p_ref > p_alt else 'ALT'
    print(f"  {r:20s}  {p_ref:>16.2f}  {p_alt:>16.2f}  {pref:>8}")

In [ ]:
# Step 7 — Visualization: HaplotypeCaller pipeline overview
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Panel A: Three-phase pipeline diagram
ax = axes[0]
ax.axis('off')
phases = [
    ('Phase 1\nActive Region\nDetection',
     'Scan BAM for\nmismatches + indels\n→ mark active windows',
     '#E3F2FD'),
    ('Phase 2\nLocal De Novo\nAssembly',
     'De Bruijn graph\nfrom active region reads\n→ enumerate haplotypes',
     '#E8F5E9'),
    ('Phase 3\nPairHMM\nGenotyping',
     'P(read | haplotype)\nfor all read×haplotype pairs\n→ genotype likelihoods',
     '#FFF3E0'),
]
for i, (title, desc, color) in enumerate(phases):
    y = 0.75 - i * 0.28
    ax.add_patch(mpatches.FancyBboxPatch(
        (0.1, y - 0.1), 0.8, 0.20,
        boxstyle='round,pad=0.02', facecolor=color, edgecolor='gray', lw=2
    ))
    ax.text(0.30, y + 0.06, title, ha='center', va='center',
            fontsize=10, fontweight='bold')
    ax.text(0.65, y + 0.05, desc, ha='center', va='center', fontsize=9)
    if i < len(phases) - 1:
        ax.annotate('', xy=(0.5, y - 0.10), xytext=(0.5, y - 0.04),
                     arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.set_title('A. HaplotypeCaller: Three-Phase Architecture', fontsize=11, fontweight='bold')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# Panel B: GVCF joint genotyping workflow
ax2 = axes[1]
ax2.axis('off')
steps = [
    ('Sample 1 BAM', '#E3F2FD'),
    ('Sample 2 BAM', '#E3F2FD'),
    ('Sample N BAM', '#E3F2FD'),
]
# BAM boxes
for i, (label, color) in enumerate(steps):
    ax2.add_patch(mpatches.FancyBboxPatch(
        (0.05 + i * 0.30, 0.72), 0.22, 0.12,
        boxstyle='round,pad=0.02', facecolor=color, edgecolor='gray', lw=1.5
    ))
    ax2.text(0.16 + i * 0.30, 0.78, label, ha='center', va='center', fontsize=9)

# Arrows to HaplotypeCaller
for i in range(3):
    ax2.annotate('', xy=(0.16 + i * 0.30, 0.62), xytext=(0.16 + i * 0.30, 0.72),
                 arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# HaplotypeCaller box
ax2.add_patch(mpatches.FancyBboxPatch(
    (0.05, 0.50), 0.90, 0.10,
    boxstyle='round,pad=0.02', facecolor='#E8F5E9', edgecolor='green', lw=2
))
ax2.text(0.50, 0.555, 'HaplotypeCaller -ERC GVCF (per sample)', ha='center', fontsize=9, fontweight='bold')

# Arrows to GVCFs
for i, label in enumerate(['Sample1.g.vcf', 'Sample2.g.vcf', 'SampleN.g.vcf']):
    ax2.annotate('', xy=(0.16 + i * 0.30, 0.44), xytext=(0.16 + i * 0.30, 0.50),
                 arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    ax2.add_patch(mpatches.FancyBboxPatch(
        (0.05 + i * 0.30, 0.34), 0.22, 0.08,
        boxstyle='round,pad=0.01', facecolor='#FFF3E0', edgecolor='gray', lw=1
    ))
    ax2.text(0.16 + i * 0.30, 0.38, label, ha='center', fontsize=8)

# GenomicsDBImport
ax2.annotate('', xy=(0.5, 0.24), xytext=(0.5, 0.34),
             arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax2.add_patch(mpatches.FancyBboxPatch(
    (0.1, 0.14), 0.80, 0.08,
    boxstyle='round,pad=0.01', facecolor='#F3E5F5', edgecolor='purple', lw=2
))
ax2.text(0.5, 0.18, 'GenomicsDBImport → GenotypeGVCFs → VCF', ha='center', fontsize=9, fontweight='bold')

ax2.set_title('B. GVCF workflow: joint genotyping across N samples', fontsize=11, fontweight='bold')
ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)

plt.suptitle('GATK HaplotypeCaller: Architecture and GVCF Workflow', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('haplotypecaller.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. A 5 bp deletion in a 30× WGS sample: draw the pileup you would see from a
   simple pileup caller vs. what HaplotypeCaller's assembly would produce.
2. Why is GVCF mode necessary for adding a new sample to an existing cohort
   without reprocessing all existing samples?
3. Explain the difference between local (HaplotypeCaller) and global
   (Mutect2) variant callers in terms of what evidence they use.

---
## Step 10 — Quiz

1. What are the three phases of HaplotypeCaller?
2. Why is local de novo assembly better than pileup for indel calling?
3. What is a GVCF and how does it differ from a standard VCF?
4. What does PairHMM compute, and what is the alternative (simpler) approach?
5. When would you use Mutect2 instead of HaplotypeCaller?

---
## Step 12 — Reflection

> *[In one paragraph: explain why a simple pileup approach systematically fails at
> indels, and how HaplotypeCaller's local assembly solves this problem.]*

---
*Next: `09_vcf_format_reading_filtering_annotation.ipynb`*